# 04 — Concise Review Artifact Reader

Use this notebook when the team wants to review output artifacts without opening many files manually.

This notebook reads the concise row packet:

```text
output/<row_id>/
├── final_row.csv
├── review_summary.md
├── review_decision.json
├── candidate_decisions.csv
└── product_coding_input.json
```


## Review flow

```mermaid
flowchart TD
    A[Set ROW_ARTIFACT_DIR] --> B[Load review_decision.json]
    B --> C[Show what was selected]
    C --> D[Show why selected]
    D --> E[Show how decided]
    E --> F[Show AI/model summary]
    F --> G[Show rejected candidates]
    G --> H[Reviewer decision]
```


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROJECT_ROOT


## Set row artifact path

Point this to one row folder generated by Notebook 01 or Notebook 02.


In [ ]:
ROW_ARTIFACT_DIR = PROJECT_ROOT / 'output' / 'PUT_ROW_ID_HERE'

review_json = ROW_ARTIFACT_DIR / 'review_decision.json'
candidate_csv = ROW_ARTIFACT_DIR / 'candidate_decisions.csv'
summary_md = ROW_ARTIFACT_DIR / 'review_summary.md'

assert review_json.exists(), f'Missing {review_json}'
assert candidate_csv.exists(), f'Missing {candidate_csv}'
assert summary_md.exists(), f'Missing {summary_md}'


In [ ]:
payload = json.loads(review_json.read_text(encoding='utf-8'))
candidates = pd.read_csv(candidate_csv)
decision = payload['decision']
checks = payload['checks']

decision


## Decision snapshot


In [ ]:
pd.DataFrame([decision]).T.rename(columns={0: 'value'})


## Gate checks


In [ ]:
pd.DataFrame([checks]).T.rename(columns={0: 'value'})


## Why selected


In [ ]:
for point in payload.get('why_selected', []):
    print('-', point)


## How decided


In [ ]:
for point in payload.get('how_decided', []):
    print('-', point)


## AI/model reasoning summary

This shows observable model/planner outputs only. It is not hidden chain-of-thought.


In [ ]:
for point in payload.get('ai_reasoning_summary', []):
    print('-', point)


## Candidate decisions

This is the table reviewers should use to see what was selected, what was rejected, and why.


In [ ]:
review_columns = [
    'rank', 'selected', 'decision', 'url', 'confidence', 'validation_status',
    'identity_status', 'exact_product_check', 'country_check', 'retailer_check',
    'scrapable', 'product_page', 'reason'
]
candidates[[c for c in review_columns if c in candidates.columns]]


## Human-readable summary file

Open this file directly when sharing the decision outside the notebook:


In [ ]:
print(summary_md)
print('
First 1200 characters:
')
print(summary_md.read_text(encoding='utf-8')[:1200])
